# Weather Forecasting using LSTM and ARIMA
### Dataset: Jena Climate (Max Planck Institute, 2009–2016)

**Run all cells top to bottom — Ctrl+F9 in Colab**

| Section | What it does |
|---|---|
| 0 | Install & import libraries |
| 1 | Download & load Jena Climate dataset |
| 2 | Exploratory Data Analysis |
| 3 | Feature engineering (uncommon features) |
| 4 | Stationarity test & ARIMA order selection |
| 5 | Train / val / test split |
| 6 | ARIMA model |
| 7 | LSTM model |
| 8 | Hybrid ARIMA + LSTM |
| 9 | Evaluation & comparison |
| 10 | Feature importance |
| 11 | Save outputs |

## Section 0 — Install & Import Libraries

In [ ]:
# Install kagglehub if not already installed (Colab may not have it)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])

In [ ]:
# Standard library
import warnings
import math
import os
warnings.filterwarnings("ignore")

# Data handling
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from keras.models import Model, Sequential
from keras.layers import (
    Input, LSTM, Dense, Dropout,
    Bidirectional, Conv1D, BatchNormalization
)
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.optimizers import Adam

# Statistical models
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Kaggle dataset downloader
import kagglehub

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("All libraries loaded successfully")

## Section 1 — Download & Load Jena Climate Dataset

In [ ]:
# Download Jena Climate dataset from Kaggle (no login needed in Colab)
path = kagglehub.dataset_download("mnassrib/jena-climate")
DATA_PATH  = os.path.join(path, "jena_climate_2009_2016.csv")
TARGET_COL = "Temperature"

print(f"Downloaded to : {DATA_PATH}")

# Jena Climate column mapping
COLUMN_MAP = {
    "Date Time"    : "datetime",
    "T (degC)"     : "Temperature",
    "rh (%)"       : "Humidity",
    "p (mbar)"     : "Pressure",
    "wv (m/s)"     : "WindSpeed",
    "wd (deg)"     : "WindBearing",
    "VPmax (mbar)" : "Visibility",
    "Tdew (degC)"  : "DewPoint",
}

# Load CSV
raw_df = pd.read_csv(DATA_PATH)

# -9999 is the sentinel value for missing wind readings in Jena dataset
raw_df.replace(-9999.0, float("nan"), inplace=True)

raw_df.rename(columns=COLUMN_MAP, inplace=True)

# Jena timestamp format is DD.MM.YYYY HH:MM:SS — needs dayfirst=True
raw_df["datetime"] = pd.to_datetime(raw_df["datetime"], dayfirst=True, utc=True)
raw_df.set_index("datetime", inplace=True)
raw_df.sort_index(inplace=True)

# Keep only the 7 base features
BASE_FEATURES = ["Temperature", "Humidity", "Pressure", "WindSpeed",
                 "WindBearing", "Visibility", "DewPoint"]
df = raw_df[[c for c in BASE_FEATURES if c in raw_df.columns]].copy()

print(f"Dataset shape  : {df.shape}")
print(f"Date range     : {df.index.min()}  ->  {df.index.max()}")
print(f"Missing values :\n{df.isnull().sum()}")
df.head()

In [ ]:
# Basic cleaning — forward-fill short gaps, drop remaining NaNs, remove duplicates
df.ffill(limit=3, inplace=True)
df.dropna(inplace=True)
df = df[~df.index.duplicated(keep="first")]

# Use every 6th row to reduce dataset to hourly (Jena is recorded every 10 min)
# This keeps ~70,000 rows which is enough and trains much faster
df = df.iloc[::6].copy()

print(f"Clean hourly dataset shape: {df.shape}")

## Section 2 — Exploratory Data Analysis

In [ ]:
# Time-series overview of all base features
fig, axes = plt.subplots(len(df.columns), 1, figsize=(14, 3 * len(df.columns)))
colors = ["steelblue", "darkorange", "seagreen", "crimson",
          "purple", "saddlebrown", "teal"]

for ax, col, color in zip(axes, df.columns, colors):
    ax.plot(df.index, df[col], linewidth=0.5, alpha=0.8, color=color)
    ax.set_title(col, fontsize=11)
    ax.set_ylabel(col)
    ax.grid(alpha=0.3)

plt.suptitle("Jena Climate — Raw Feature Time Series", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(9, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", linewidths=0.5, vmin=-1, vmax=1)
plt.title("Pearson Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution and daily trend of target variable
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df[TARGET_COL].plot(kind="hist", bins=60, ax=axes[0],
                    color="steelblue", edgecolor="white")
axes[0].set_title(f"Distribution of {TARGET_COL}")
axes[0].set_xlabel("deg C")

df[TARGET_COL].resample("D").mean().plot(ax=axes[1],
                                          color="darkorange", linewidth=0.8)
axes[1].set_title(f"Daily Mean {TARGET_COL} Over Time")
axes[1].set_ylabel("deg C")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Section 3 — Uncommon Feature Engineering

| Feature Group | Features Added |
|---|---|
| Thermodynamic | HeatIndex, Humidex, DewPointDepression, Enthalpy, VaporPressureDeficit |
| Wind | WindChill, WindU, WindV, BeaufortScale |
| Pressure dynamics | PressureTendency3h, PressureTendency6h, PressureAnomaly24h |
| Fourier time | SinHour, CosHour, SinDOY, CosDOY |
| Lag features | Temp_lag1h to Temp_lag24h |
| Rolling stats | TempRollMean/Std/Min/Max over 6h and 24h |

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds 25+ uncommon meteorological and temporal features.
    All formulas are physically grounded.
    Input  : pd.DataFrame with base weather columns
    Output : pd.DataFrame with all engineered columns added
    """
    feat = df.copy()

    # Shorthand aliases
    T  = feat["Temperature"]   # deg C
    H  = feat["Humidity"]      # 0-100 %
    P  = feat["Pressure"]      # mbar
    WS = feat["WindSpeed"]     # m/s  (Jena uses m/s not km/h)
    WB = feat["WindBearing"]   # degrees
    DP = feat["DewPoint"]      # deg C

    # Convert WindSpeed m/s -> km/h for formulas that expect km/h
    WS_kmh = WS * 3.6

    # ── Group 1: Thermodynamic indices ───────────────────────────────────────

    # Heat Index (Steadman 1979) — felt temperature in heat
    T_f = T * 9/5 + 32
    feat["HeatIndex"] = (
        -42.379
        + 2.04901523   * T_f
        + 10.14333127  * H
        - 0.22475541   * T_f * H
        - 0.00683783   * T_f**2
        - 0.05481717   * H**2
        + 0.00122874   * T_f**2 * H
        + 0.00085282   * T_f * H**2
        - 0.00000199   * T_f**2 * H**2
    )
    feat["HeatIndex"] = (feat["HeatIndex"] - 32) * 5/9   # back to deg C

    # Vapour pressure at dew point — reused by Humidex and Enthalpy
    e_dp = 6.11 * np.exp(5417.7530 * (1/273.16 - 1/(273.16 + DP)))

    # Humidex — Canadian discomfort index
    feat["Humidex"] = T + 0.5555 * (e_dp - 10)

    # Dew-Point Depression — how far the air is from saturation
    feat["DewPointDepression"] = T - DP

    # Specific Enthalpy of moist air (kJ/kg dry air)
    W = 0.622 * (e_dp / (P - e_dp))   # mixing ratio kg/kg
    feat["Enthalpy"] = 1.006 * T + W * (2501 + 1.86 * T)

    # Vapor Pressure Deficit — how thirsty the atmosphere is
    e_sat = 6.112 * np.exp(17.67 * T / (T + 243.5))
    e_act = (H / 100) * e_sat
    feat["VaporPressureDeficit"] = e_sat - e_act

    # ── Group 2: Wind-based features ─────────────────────────────────────────

    # Wind Chill (Environment Canada formula — uses km/h)
    feat["WindChill"] = (
        13.12 + 0.6215 * T
        - 11.37 * WS_kmh**0.16
        + 0.3965 * T * WS_kmh**0.16
    )

    # U and V wind vector components (fixes circular angle problem)
    WB_rad = np.deg2rad(WB)
    feat["WindU"] = -WS * np.sin(WB_rad)   # zonal (east-west)
    feat["WindV"] = -WS * np.cos(WB_rad)   # meridional (north-south)

    # Beaufort scale (0-12 categorical wind strength)
    beaufort_bins   = [0, 0.3, 1.6, 3.4, 5.5, 8.0, 10.8,
                       13.9, 17.2, 20.8, 24.5, 28.5, 32.7, np.inf]
    beaufort_labels = list(range(13))
    feat["BeaufortScale"] = pd.cut(
        WS, bins=beaufort_bins, labels=beaufort_labels, right=False
    ).astype(float)

    # ── Group 3: Pressure dynamics ────────────────────────────────────────────

    feat["PressureTendency3h"]  = P.diff(periods=3)
    feat["PressureTendency6h"]  = P.diff(periods=6)
    feat["PressureAnomaly24h"]  = P - P.rolling(window=24, min_periods=1).mean()

    # ── Group 4: Fourier time encodings (cyclical, no discontinuity) ──────────

    hour_of_day = feat.index.hour
    day_of_year = feat.index.dayofyear

    feat["SinHour"] = np.sin(2 * np.pi * hour_of_day / 24)
    feat["CosHour"] = np.cos(2 * np.pi * hour_of_day / 24)
    feat["SinDOY"]  = np.sin(2 * np.pi * day_of_year / 365)
    feat["CosDOY"]  = np.cos(2 * np.pi * day_of_year / 365)

    # ── Group 5: Lag features (autoregressive memory) ─────────────────────────

    for lag in [1, 3, 6, 12, 24]:
        feat[f"Temp_lag{lag}h"] = T.shift(lag)

    # ── Group 6: Rolling statistics ───────────────────────────────────────────

    for window in [6, 24]:
        feat[f"TempRollMean{window}h"] = T.rolling(window, min_periods=1).mean()
        feat[f"TempRollStd{window}h"]  = T.rolling(window, min_periods=1).std()
        feat[f"TempRollMin{window}h"]  = T.rolling(window, min_periods=1).min()
        feat[f"TempRollMax{window}h"]  = T.rolling(window, min_periods=1).max()

    # Drop NaN rows introduced by lag and diff operations
    feat.dropna(inplace=True)

    return feat


df_feat = engineer_features(df)

print(f"Features before engineering : {df.shape[1]}")
print(f"Features after  engineering : {df_feat.shape[1]}")
print(f"Rows after dropna           : {len(df_feat):,}")
print(f"\nNew features added:")
new_cols = [c for c in df_feat.columns if c not in df.columns]
for c in new_cols:
    print(f"  {c}")

In [ ]:
# Visualise uncommon engineered features
uncommon_to_plot = ["HeatIndex", "Humidex", "Enthalpy",
                    "VaporPressureDeficit", "PressureTendency3h", "WindChill"]

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, col in zip(axes, uncommon_to_plot):
    ax.plot(df_feat.index, df_feat[col], linewidth=0.5, alpha=0.8)
    ax.set_title(col, fontsize=11)
    ax.grid(alpha=0.3)

plt.suptitle("Engineered Features — Time Series View", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Section 4 — Stationarity Test & ARIMA Order Selection

In [ ]:
def adf_test(series: pd.Series, label: str = "Series") -> None:
    """
    Augmented Dickey-Fuller test for stationarity.
    p < 0.05  -> stationary (safe for ARIMA without differencing)
    p >= 0.05 -> non-stationary (needs differencing, increase d)
    """
    result = adfuller(series.dropna())
    print(f"ADF Test: {label}")
    print(f"  ADF Statistic : {result[0]:.4f}")
    print(f"  p-value       : {result[1]:.4f}")
    print(f"  Critical vals : { {k: f'{v:.3f}' for k, v in result[4].items()} }")
    if result[1] < 0.05:
        print("  RESULT: Stationary (reject H0) -> d=0 is fine")
    else:
        print("  RESULT: Non-stationary -> use d=1 in ARIMA")
    print()


adf_test(df_feat[TARGET_COL], label="Temperature (raw)")
adf_test(df_feat[TARGET_COL].diff().dropna(), label="Temperature (1st difference)")

In [ ]:
# ACF and PACF plots to choose ARIMA (p, d, q)
# p -> where PACF cuts off, q -> where ACF cuts off
sample = df_feat[TARGET_COL].dropna().iloc[:3000]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(sample,  lags=48, ax=axes[0], title="ACF — Temperature")
plot_pacf(sample, lags=48, ax=axes[1], title="PACF — Temperature", method="ywm")
plt.tight_layout()
plt.show()

## Section 5 — Train / Validation / Test Split

In [ ]:
# Chronological split — NO random shuffling for time-series
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# Test = remaining 15%

n       = len(df_feat)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)

train_df = df_feat.iloc[:n_train]
val_df   = df_feat.iloc[n_train : n_train + n_val]
test_df  = df_feat.iloc[n_train + n_val:]

print(f"Total samples : {n:,}")
print(f"Train         : {len(train_df):,}  ({TRAIN_RATIO*100:.0f}%)")
print(f"Validation    : {len(val_df):,}  ({VAL_RATIO*100:.0f}%)")
print(f"Test          : {len(test_df):,}  ({(1-TRAIN_RATIO-VAL_RATIO)*100:.0f}%)")

## Section 6 — ARIMA Model

In [ ]:
# SARIMA hyperparameters
SARIMA_P, SARIMA_D, SARIMA_Q = 2, 0, 2
SEASONAL_P, SEASONAL_D, SEASONAL_Q, SEASONAL_S = 1, 1, 1, 24

# Train/Test series
sarima_train_series = train_df[TARGET_COL]
sarima_test_series  = pd.concat([val_df[TARGET_COL], test_df[TARGET_COL]])

print(
    f"Fitting SARIMA({SARIMA_P},{SARIMA_D},{SARIMA_Q})"
    f"({SEASONAL_P},{SEASONAL_D},{SEASONAL_Q},{SEASONAL_S}) "
    f"on {len(sarima_train_series):,} points..."
)

# Build SARIMA model
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_model = SARIMAX(
    sarima_train_series,
    order=(SARIMA_P, SARIMA_D, SARIMA_Q),
    seasonal_order=(SEASONAL_P, SEASONAL_D, SEASONAL_Q, SEASONAL_S),
    enforce_stationarity=False,
    enforce_invertibility=False
)

# Fit model
sarima_fit = sarima_model.fit()

# Summary
print(sarima_fit.summary())

In [ ]:
# SARIMA forecast on test set
sarima_forecast = sarima_fit.forecast(
    steps=len(sarima_test_series)
)
sarima_forecast.index = sarima_test_series.index

# SARIMA residuals on training data
sarima_train_residuals = sarima_fit.resid

# Error metrics
sarima_rmse = math.sqrt(
    mean_squared_error(sarima_test_series, sarima_forecast)
)

sarima_mae = mean_absolute_error(
    sarima_test_series,
    sarima_forecast
)

print(f"SARIMA Test RMSE : {sarima_rmse:.4f} deg C")
print(f"SARIMA Test MAE  : {sarima_mae:.4f} deg C")

# Plot
plt.figure(figsize=(14, 4))

plt.plot(
    sarima_test_series.iloc[:200].values,
    label="Actual",
)

plt.plot(
    sarima_forecast.iloc[:200].values,
    label="SARIMA Forecast",
    linestyle="--"
)

plt.title("SARIMA Forecast vs Actual (first 200 test steps)")
plt.xlabel("Time Step (hours)")
plt.ylabel("Temperature (deg C)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Section 7 — LSTM Model (Standalone)
Architecture: Conv1D -> Bidirectional LSTM -> Dense

In [ ]:
# Normalise using train statistics only — prevents data leakage
ALL_FEATURES = df_feat.columns.tolist()
TARGET_IDX   = ALL_FEATURES.index(TARGET_COL)

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_df[ALL_FEATURES])
val_scaled   = scaler.transform(val_df[ALL_FEATURES])
test_scaled  = scaler.transform(test_df[ALL_FEATURES])

print(f"Number of input features : {len(ALL_FEATURES)}")
print(f"Target column index      : {TARGET_IDX}  ({TARGET_COL})")

In [ ]:
def make_sequences(data: np.ndarray, lookback: int, target_idx: int):
    """
    Slides a window of length `lookback` across the 2D scaled array.
    X shape: (samples, lookback, n_features)
    y shape: (samples,)  — target value at next time step
    """
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback : i, :])
        y.append(data[i, target_idx])
    return np.array(X), np.array(y)


LOOKBACK = 24   # use past 24 hours to predict next hour

X_train, y_train = make_sequences(train_scaled, LOOKBACK, TARGET_IDX)
X_val,   y_val   = make_sequences(val_scaled,   LOOKBACK, TARGET_IDX)
X_test,  y_test  = make_sequences(test_scaled,  LOOKBACK, TARGET_IDX)

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")

In [ ]:
def build_lstm_model(input_shape: tuple) -> keras.Model:
    """
    Conv1D -> BatchNorm -> BiLSTM(128) -> BiLSTM(64) -> Dropout -> Dense(64) -> Dense(1)
    Bidirectional LSTM reads the input sequence in both directions for richer context.
    """
    inputs = Input(shape=input_shape, name="input")

    x = Conv1D(filters=64, kernel_size=3, activation="relu",
               padding="same", name="conv1d")(inputs)
    x = BatchNormalization(name="bn1")(x)

    x = Bidirectional(
        LSTM(128, return_sequences=True, dropout=0.1, recurrent_dropout=0.1),
        name="bilstm1"
    )(x)

    x = Bidirectional(
        LSTM(64, return_sequences=False, dropout=0.1),
        name="bilstm2"
    )(x)

    x = Dropout(0.2, name="dropout1")(x)
    x = Dense(64, activation="relu", name="dense1")(x)
    x = Dense(32, activation="relu", name="dense2")(x)
    outputs = Dense(1, name="output")(x)

    model = Model(inputs, outputs, name="BiLSTM_Weather")
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model


lstm_model = build_lstm_model(input_shape=(LOOKBACK, X_train.shape[2]))
lstm_model.summary()

In [ ]:
# Callbacks
callbacks = [
    EarlyStopping(
        monitor="val_loss", patience=8,
        restore_best_weights=True, verbose=1
    ),
    ModelCheckpoint(
        filepath="best_lstm_model.keras",
        monitor="val_loss", save_best_only=True, verbose=0
    ),
    ReduceLROnPlateau(
        monitor="val_loss", factor=0.5,
        patience=4, min_lr=1e-6, verbose=1
    ),
]

# Train
BATCH_SIZE = 128
EPOCHS     = 10

history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

print("Training complete")

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history["loss"],     label="Train Loss",      color="royalblue")
axes[0].plot(history.history["val_loss"], label="Validation Loss", color="orangered")
axes[0].set_title("MSE Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history["mae"],     label="Train MAE",      color="royalblue")
axes[1].plot(history.history["val_mae"], label="Validation MAE", color="orangered")
axes[1].set_title("Mean Absolute Error")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("LSTM Training History", fontsize=13)
plt.tight_layout()
plt.show()

## Section 8 — Hybrid ARIMA + LSTM

ARIMA captures linear patterns. Its residuals contain the non-linear signal ARIMA missed.
A second LSTM is trained on those residuals.
Final prediction = ARIMA forecast + LSTM(residuals)

In [ ]:
def make_residual_sequences(residuals: np.ndarray, lookback: int):
    """
    Same sliding-window logic as make_sequences() but for 1D residual array.
    Output: X shape (samples, lookback, 1), y shape (samples,)
    """
    X, y = [], []

    for i in range(lookback, len(residuals)):
        X.append(residuals[i - lookback : i])
        y.append(residuals[i])

    return np.array(X).reshape(-1, lookback, 1), np.array(y)


def build_residual_lstm(lookback: int) -> keras.Model:
    """
    Lightweight LSTM trained only on SARIMA residuals.
    Smaller than the main model — only needs to capture leftover non-linear signal.
    """

    model = Sequential([
        Input(shape=(lookback, 1)),

        LSTM(64, return_sequences=True, dropout=0.1),

        LSTM(32, return_sequences=False),

        Dropout(0.2),

        Dense(16, activation="relu"),

        Dense(1)
    ], name="Residual_LSTM")

    model.compile(
        optimizer=Adam(1e-3),
        loss="mse",
        metrics=["mae"]
    )

    return model


# =========================
# SARIMA Residual Processing
# =========================

# Residuals from SARIMA model
residuals_train = sarima_fit.resid.values

# Scale residuals separately
residual_scaler = MinMaxScaler(feature_range=(-1, 1))

residuals_scaled = residual_scaler.fit_transform(
    residuals_train.reshape(-1, 1)
).flatten()

# Sequence generation
RESIDUAL_LOOKBACK = 12

X_res_train, y_res_train = make_residual_sequences(
    residuals_scaled,
    RESIDUAL_LOOKBACK
)

print(f"Residual LSTM input  : {X_res_train.shape}")
print(f"Residual LSTM target : {y_res_train.shape}")

# Build residual LSTM
residual_lstm = build_residual_lstm(RESIDUAL_LOOKBACK)

# Model summary
residual_lstm.summary()

In [ ]:
res_callbacks = [
    EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
]

res_history = residual_lstm.fit(
    X_res_train, y_res_train,
    epochs=50,
    batch_size=64,
    validation_split=0.15,
    callbacks=res_callbacks,
    verbose=1,
)

print("Residual LSTM training complete")

## Section 9 — Evaluation & Comparison

In [ ]:
def inverse_target(scaled_values: np.ndarray, scaler: MinMaxScaler,
                   n_features: int, target_idx: int) -> np.ndarray:
    """
    Reverses MinMaxScaler on the target column only.
    Creates a dummy full-width matrix, inserts predictions at target_idx,
    runs inverse_transform, then extracts that column.
    Returns real-world temperature values in deg C.
    """

    dummy = np.zeros((len(scaled_values), n_features))
    dummy[:, target_idx] = scaled_values.flatten()

    return scaler.inverse_transform(dummy)[:, target_idx]


def evaluate(y_true: np.ndarray, y_pred: np.ndarray, label: str) -> dict:
    """
    Computes RMSE, MAE, MAPE, R2 and prints a one-line summary.
    """

    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)

    mape = np.mean(
        np.abs((y_true - y_pred) / (y_true + 1e-8))
    ) * 100

    r2 = r2_score(y_true, y_pred)

    print(
        f"  {label:<25}  "
        f"RMSE={rmse:.4f}  "
        f"MAE={mae:.4f}  "
        f"MAPE={mape:.2f}%  "
        f"R2={r2:.4f}"
    )

    return {
        "Model": label,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    }


# =========================
# Prediction Evaluation
# =========================

n_features = X_train.shape[2]

# -------------------------
# Standalone LSTM predictions
# -------------------------

lstm_pred_scaled = lstm_model.predict(
    X_test,
    verbose=0
).flatten()

lstm_pred = inverse_target(
    lstm_pred_scaled,
    scaler,
    n_features,
    TARGET_IDX
)

y_test_actual = inverse_target(
    y_test,
    scaler,
    n_features,
    TARGET_IDX
)

# -------------------------
# SARIMA predictions aligned
# -------------------------

sarima_test_aligned = sarima_forecast.values[
    -len(y_test_actual):
]

# -------------------------
# Residual generation
# -------------------------

test_residuals_raw = (
    y_test_actual - sarima_test_aligned
)

test_residuals_scaled = residual_scaler.transform(
    test_residuals_raw.reshape(-1, 1)
).flatten()

# -------------------------
# Residual LSTM prediction
# -------------------------

X_res_test, y_res_test = make_residual_sequences(
    test_residuals_scaled,
    RESIDUAL_LOOKBACK
)

residual_pred_scaled = residual_lstm.predict(
    X_res_test,
    verbose=0
).flatten()

residual_pred = residual_scaler.inverse_transform(
    residual_pred_scaled.reshape(-1, 1)
).flatten()

# -------------------------
# Align lengths
# -------------------------

min_len = min(
    len(y_test_actual),
    len(sarima_test_aligned),
    len(residual_pred)
)

y_actual = y_test_actual[-min_len:]

sarima_pred_test = sarima_test_aligned[-min_len:]

lstm_pred_test = lstm_pred[-min_len:]

# Final Hybrid Forecast
hybrid_pred = (
    sarima_pred_test +
    residual_pred[-min_len:]
)

# =========================
# Evaluation
# =========================

print(f"Aligned test length : {min_len:,}")
print()

print("Test Set Performance")
print("-" * 70)

results = [

    evaluate(
        y_actual,
        sarima_pred_test,
        "SARIMA"
    ),

    evaluate(
        y_actual,
        lstm_pred_test,
        "LSTM (standalone)"
    ),

    evaluate(
        y_actual,
        hybrid_pred,
        "Hybrid SARIMA+LSTM"
    ),
]

print("-" * 70)

results_df = pd.DataFrame(results).set_index("Model")

results_df

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
palette   = ["#4C72B0", "#DD8452", "#55A868"]

for ax, metric in zip(axes, ["RMSE", "MAE", "R2"]):
    bars = results_df[metric].plot(
        kind="bar", ax=ax, color=palette, edgecolor="white", width=0.6
    )
    ax.set_title(metric, fontsize=12)
    ax.set_xticklabels(results_df.index, rotation=20, ha="right")
    ax.grid(axis="y", alpha=0.4)
    for bar in ax.patches:
        ax.annotate(f"{bar.get_height():.3f}",
                    (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    ha="center", va="bottom", fontsize=9)

plt.suptitle("Model Comparison on Test Set", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Forecast vs Actual — all three models

PLOT_N = 300

plt.figure(figsize=(15, 5))

# Actual values
plt.plot(
    y_actual[:PLOT_N],
    label="Actual",
    color="black",
    linewidth=1.5
)

# SARIMA forecast
plt.plot(
    sarima_pred_test[:PLOT_N],
    label="SARIMA",
    color="#4C72B0",
    linestyle="--",
    alpha=0.8
)

# Standalone LSTM
plt.plot(
    lstm_pred_test[:PLOT_N],
    label="LSTM",
    color="#DD8452",
    linestyle="-.",
    alpha=0.8
)

# Hybrid forecast
plt.plot(
    hybrid_pred[:PLOT_N],
    label="Hybrid SARIMA+LSTM",
    color="#55A868",
    linestyle=":",
    linewidth=2
)

plt.title(
    f"Temperature Forecast — First {PLOT_N} Test Steps",
    fontsize=13
)

plt.xlabel("Time Step (hours)")
plt.ylabel("Temperature (deg C)")

plt.legend(fontsize=11)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()

In [ ]:
# Scatter: Predicted vs Actual for all three models

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models_preds = [
    ("SARIMA",               sarima_pred_test, "#4C72B0"),
    ("LSTM",                 lstm_pred_test,   "#DD8452"),
    ("Hybrid SARIMA+LSTM",  hybrid_pred,      "#55A868"),
]

for ax, (name, pred, color) in zip(axes, models_preds):

    ax.scatter(
        y_actual,
        pred,
        alpha=0.2,
        s=4,
        color=color
    )

    # Perfect prediction diagonal
    lim = [
        min(y_actual.min(), pred.min()),
        max(y_actual.max(), pred.max())
    ]

    ax.plot(
        lim,
        lim,
        "k--",
        linewidth=1
    )

    ax.set_title(name)

    ax.set_xlabel("Actual (deg C)")
    ax.set_ylabel("Predicted (deg C)")

    ax.grid(alpha=0.3)

plt.suptitle(
    "Predicted vs Actual Temperature",
    fontsize=13
)

plt.tight_layout()

plt.show()

## Section 10 — Permutation Feature Importance

In [ ]:
def permutation_importance(model, X: np.ndarray, y: np.ndarray,
                            feature_names: list, n_repeats: int = 2) -> pd.DataFrame:
    """
    Estimates how much each feature contributes to LSTM accuracy.
    For each feature: shuffle it randomly across test samples,
    measure how much MSE rises. Higher rise = more important feature.
    n_repeats=2 keeps runtime reasonable for large feature sets.
    """
    baseline_mse = mean_squared_error(y, model.predict(X, verbose=0).flatten())
    importances  = []

    for feat_idx, feat_name in enumerate(feature_names):
        drops = []
        for _ in range(n_repeats):
            X_permuted = X.copy()
            perm = np.random.permutation(X_permuted.shape[0])
            X_permuted[:, :, feat_idx] = X_permuted[perm, :, feat_idx]
            perm_mse = mean_squared_error(y, model.predict(X_permuted, verbose=0).flatten())
            drops.append(perm_mse - baseline_mse)
        importances.append({"Feature": feat_name, "Importance": np.mean(drops)})

    return pd.DataFrame(importances).sort_values("Importance", ascending=False)


print("Computing permutation importance (takes a few minutes)...")
importance_df = permutation_importance(
    lstm_model, X_test, y_test, ALL_FEATURES, n_repeats=2
)

# Plot top 20 features
top_n = 20
plt.figure(figsize=(10, 7))
plt.barh(
    importance_df["Feature"].head(top_n)[::-1],
    importance_df["Importance"].head(top_n)[::-1],
    color="steelblue", edgecolor="white"
)
plt.xlabel("Mean MSE Increase (higher = more important)")
plt.title(f"Top {top_n} Feature Importances (Permutation Method)", fontsize=13)
plt.grid(axis="x", alpha=0.4)
plt.tight_layout()
plt.show()

importance_df.head(top_n)

## Section 11 — Save All Outputs

In [ ]:
# =========================
# Save Performance Summary
# =========================

results_df.to_csv(
    "model_performance_summary.csv"
)

print("Saved: model_performance_summary.csv")


# =========================
# Save Predictions
# =========================

predictions_df = pd.DataFrame({

    "Actual": y_actual,

    "SARIMA": sarima_pred_test,

    "LSTM": lstm_pred_test,

    "Hybrid_SARIMA_LSTM": hybrid_pred,
})

predictions_df.to_csv(
    "test_predictions.csv",
    index=False
)

print("Saved: test_predictions.csv")


# =========================
# Save Feature Importance
# =========================

importance_df.to_csv(
    "feature_importance.csv",
    index=False
)

print("Saved: feature_importance.csv")


# =========================
# Save Trained LSTM Model
# =========================

lstm_model.save(
    "lstm_weather_model.keras"
)

print("Saved: lstm_weather_model.keras")


# =========================
# Final Summary
# =========================

print("\nAll outputs saved successfully")

print("\nFinal results:")

print(
    results_df.round(4)
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')